# Modelo Simplificado + Análisis de Falsos Negativos
Solo las 6 features con señal real — sin features temporales de laboratorio  
Split estratificado 80/20 con shuffle

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score
)
import matplotlib.pyplot as plt

# Usamos features_fe (sin temporales de laboratorio)
DELTA_FE_PATH = "/Volumes/workspace/default/network_data/features_fe/"

df = spark.read.format("delta").load(DELTA_FE_PATH)
print(f"Total ventanas : {df.count():,}")

## 1 — Solo las 6 features con señal real

In [0]:
# Las únicas features que el modelo usa de verdad según importancia
# Las temporales se descartan por ser artefactos del laboratorio
FEATURE_COLS = [
    "avg_connection_duration_ms",   # 0.372 — la más importante
    "max_connection_duration_ms",   # 0.274
    "packets_per_ms",               # 0.178 — feature creada en FE
    "duration_spread",              # 0.093 — feature creada en FE
    "is_short_duration",            # 0.045 — feature creada en FE
    "duration_ratio",               # 0.036 — feature creada en FE
]

print(f"Features seleccionadas: {len(FEATURE_COLS)}")
for f in FEATURE_COLS:
    print(f"  - {f}")

In [0]:
# Traer a pandas con window_start para el análisis de FN posterior
pdf = (
    df.select(FEATURE_COLS + ["label", "window_start"])
      .toPandas()
)

pdf[FEATURE_COLS] = pdf[FEATURE_COLS].fillna(0).astype(float)
pdf["label"]      = pdf["label"].astype(int)

print(f"Shape      : {pdf.shape}")
print(f"Normal (0) : {(pdf['label']==0).sum():,}")
print(f"Ataque (1) : {(pdf['label']==1).sum():,}")

## 2 — Split estratificado con shuffle

In [0]:
X = pdf[FEATURE_COLS].values
y = pdf["label"].values
idx = pdf.index.values  # guardamos índices para recuperar window_start en FN

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, idx,
    test_size=0.2,
    shuffle=True,
    stratify=y,
    random_state=42
)

n_normal = (y_train == 0).sum()
n_ataque = (y_train == 1).sum()
weight_ataque = round(n_normal / n_ataque, 2)

print(f"Train : {len(X_train):,}  |  Ataques: {y_train.sum():,}  ({y_train.mean()*100:.2f}%)")
print(f"Test  : {len(X_test):,}   |  Ataques: {y_test.sum():,}  ({y_test.mean()*100:.2f}%)")
print(f"Peso clase Ataque : {weight_ataque}")

## 3 — Entrenar Random Forest simplificado

In [0]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=5,
    class_weight={0: 1.0, 1: weight_ataque},
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("Entrenando con 6 features...")
rf.fit(X_train, y_train)
print("✅ Entrenamiento completado")

## 4 — Evaluación

In [0]:
y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print("=" * 50)
print("MÉTRICAS — MODELO SIMPLIFICADO (6 features)")
print("=" * 50)
print(classification_report(
    y_test, y_pred,
    target_names=["Normal (0)", "Ataque (1)"]
))

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"AUC-ROC                : {roc_auc_score(y_test, y_proba):.4f}")
print(f"AUC-PR                 : {average_precision_score(y_test, y_proba):.4f}")
print("-" * 50)
print(f"Verdaderos Negativos   : {tn:,}  — Normales correctos")
print(f"Falsos Positivos       : {fp:,}  — Normales como Ataque")
print(f"Falsos Negativos       : {fn:,}  — Ataques no detectados ⚠️")
print(f"Verdaderos Positivos   : {tp:,}  — Ataques detectados")
print("-" * 50)
print(f"Tasa detección ataques : {tp/(tp+fn)*100:.2f}%")
print(f"Tasa falsa alarma      : {fp/(fp+tn)*100:.2f}%")
print("=" * 50)

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im = axes[0].imshow(cm, cmap="Blues")
plt.colorbar(im, ax=axes[0])
labels_cm = ["Normal (0)", "Ataque (1)"]
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(labels_cm)
axes[0].set_yticklabels(labels_cm)
axes[0].set_xlabel("Predicción", fontsize=12)
axes[0].set_ylabel("Real", fontsize=12)
axes[0].set_title("Matriz de Confusión — Modelo Simplificado",
                  fontsize=12, fontweight="bold")
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, f"{cm[i,j]:,}",
                     ha="center", va="center",
                     color="white" if cm[i,j] > cm.max()/2 else "black",
                     fontsize=14, fontweight="bold")

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color="#4C8BF5", lw=2,
             label=f"AUC = {roc_auc_score(y_test, y_proba):.4f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("Tasa Falsos Positivos")
axes[1].set_ylabel("Tasa Verdaderos Positivos")
axes[1].set_title("Curva ROC", fontsize=12, fontweight="bold")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5 — Análisis de Falsos Negativos

In [0]:
# Reconstruir dataframe de test con predicciones y timestamps
df_test = pdf.iloc[idx_test].copy()
df_test["y_pred"]  = y_pred
df_test["y_proba"] = y_proba

# Separar FN (ataques reales que el modelo clasifica como Normal)
df_fn = df_test[(df_test["label"] == 1) & (df_test["y_pred"] == 0)].copy()
df_tp = df_test[(df_test["label"] == 1) & (df_test["y_pred"] == 1)].copy()

print(f"Ataques en test         : {(df_test['label']==1).sum():,}")
print(f"Detectados (TP)         : {len(df_tp):,}  ({len(df_tp)/(len(df_tp)+len(df_fn))*100:.1f}%)")
print(f"No detectados (FN)      : {len(df_fn):,}  ({len(df_fn)/(len(df_tp)+len(df_fn))*100:.1f}%)")
print(f"\nProbabilidad media FN   : {df_fn['y_proba'].mean():.4f}")
print(f"Probabilidad media TP   : {df_tp['y_proba'].mean():.4f}")

In [0]:
# Comparar las 6 features entre TP y FN
print("Comparación de features — TP (detectados) vs FN (no detectados):")
print("=" * 70)
comparison = pd.DataFrame({
    "feature": FEATURE_COLS,
    "media_TP": [df_tp[f].mean() for f in FEATURE_COLS],
    "media_FN": [df_fn[f].mean() for f in FEATURE_COLS],
})
comparison["diferencia_%"] = (
    (comparison["media_FN"] - comparison["media_TP"])
    / comparison["media_TP"].abs() * 100
).round(1)
print(comparison.to_string(index=False))

In [0]:
# Distribución de avg_connection_duration_ms: Normal vs TP vs FN
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_normal_test = df_test[df_test["label"] == 0]

# Histograma de avg_connection_duration_ms
axes[0].hist(df_normal_test["avg_connection_duration_ms"] / 1e6,
             bins=60, alpha=0.5, color="#4C8BF5", label="Normal", density=True)
axes[0].hist(df_tp["avg_connection_duration_ms"] / 1e6,
             bins=60, alpha=0.6, color="#2ECC71", label="Ataque detectado (TP)", density=True)
axes[0].hist(df_fn["avg_connection_duration_ms"] / 1e6,
             bins=60, alpha=0.8, color="#E8453C", label="Ataque NO detectado (FN)", density=True)
axes[0].set_title("avg_connection_duration_ms\nNormal vs TP vs FN", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Duración media (segundos)")
axes[0].set_ylabel("Densidad")
axes[0].legend(fontsize=9)

# Histograma de packets_per_ms
axes[1].hist(df_normal_test["packets_per_ms"],
             bins=60, alpha=0.5, color="#4C8BF5", label="Normal", density=True)
axes[1].hist(df_tp["packets_per_ms"],
             bins=60, alpha=0.6, color="#2ECC71", label="Ataque detectado (TP)", density=True)
axes[1].hist(df_fn["packets_per_ms"],
             bins=60, alpha=0.8, color="#E8453C", label="Ataque NO detectado (FN)", density=True)
axes[1].set_title("packets_per_ms\nNormal vs TP vs FN", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Paquetes por ms")
axes[1].set_ylabel("Densidad")
axes[1].legend(fontsize=9)

plt.suptitle("¿Por qué el modelo no detecta los FN?", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [0]:
# Distribución temporal de FN — ¿se concentran en algún momento?
df_fn["hora"] = pd.to_datetime(df_fn["window_start"]).dt.hour
df_tp["hora"] = pd.to_datetime(df_tp["window_start"]).dt.hour

fig, ax = plt.subplots(figsize=(12, 4))
hora_fn = df_fn["hora"].value_counts().sort_index()
hora_tp = df_tp["hora"].value_counts().sort_index()

x = np.arange(24)
w = 0.4
ax.bar(x - w/2, [hora_tp.get(h, 0) for h in x], w,
       color="#2ECC71", alpha=0.8, label="TP (detectados)")
ax.bar(x + w/2, [hora_fn.get(h, 0) for h in x], w,
       color="#E8453C", alpha=0.8, label="FN (no detectados)")
ax.set_xticks(x)
ax.set_xticklabels([f"{h:02d}h" for h in x], rotation=45, fontsize=8)
ax.set_title("Distribución temporal — TP vs FN por hora",
             fontsize=12, fontweight="bold")
ax.set_ylabel("Nº ventanas")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Resumen por hora con tasa de detección
print("\nTasa de detección por hora:")
fn_rate = pd.DataFrame({
    "hora": range(24),
    "TP":   [hora_tp.get(h, 0) for h in range(24)],
    "FN":   [hora_fn.get(h, 0) for h in range(24)],
})
fn_rate["total_ataques"] = fn_rate["TP"] + fn_rate["FN"]
fn_rate["tasa_deteccion_%"] = (
    fn_rate["TP"] / fn_rate["total_ataques"] * 100
).round(1).fillna(0)
print(fn_rate[fn_rate["total_ataques"] > 0].to_string(index=False))

## 6 — Umbral óptimo

In [0]:
thresholds_range = np.arange(0.1, 0.9, 0.05)
results = []

for t in thresholds_range:
    y_pred_t = (y_proba >= t).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    results.append({
        "threshold":        round(t, 2),
        "recall_ataque":    round(tp_t / (tp_t + fn_t) * 100, 2),
        "precision_ataque": round(tp_t / (tp_t + fp_t) * 100, 2) if (tp_t + fp_t) > 0 else 0,
        "falsa_alarma":     round(fp_t / (fp_t + tn_t) * 100, 2),
        "f1_ataque":        round(2 * tp_t / (2 * tp_t + fp_t + fn_t) * 100, 2) if (2*tp_t + fp_t + fn_t) > 0 else 0
    })

df_thresh = pd.DataFrame(results)
print(df_thresh.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_thresh["threshold"], df_thresh["recall_ataque"],
        color="#E8453C", lw=2, label="Recall Ataque (%)")
ax.plot(df_thresh["threshold"], df_thresh["precision_ataque"],
        color="#4C8BF5", lw=2, label="Precision Ataque (%)")
ax.plot(df_thresh["threshold"], df_thresh["falsa_alarma"],
        color="gray", lw=1.5, linestyle="--", label="Falsa Alarma (%)")
ax.plot(df_thresh["threshold"], df_thresh["f1_ataque"],
        color="#F59E0B", lw=2, label="F1 Ataque (%)")
ax.axvline(0.5, color="black", linestyle=":", lw=1, label="Umbral default (0.5)")
ax.set_xlabel("Umbral de clasificación")
ax.set_ylabel("%")
ax.set_title("Métricas vs Umbral — Modelo Simplificado",
             fontsize=12, fontweight="bold")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best = df_thresh.loc[df_thresh["f1_ataque"].idxmax()]
print(f"\nUmbral óptimo (F1) : {best['threshold']}")
print(f"  Recall Ataque    : {best['recall_ataque']}%")
print(f"  Precision Ataque : {best['precision_ataque']}%")
print(f"  Falsa Alarma     : {best['falsa_alarma']}%")
print(f"  F1 Ataque        : {best['f1_ataque']}%")

## 7 — Resumen final

In [0]:
print("=" * 55)
print("RESUMEN — MODELO SIMPLIFICADO (6 features)")
print("=" * 55)
print(f"  Features usadas      : {len(FEATURE_COLS)}")
for f in FEATURE_COLS:
    imp = rf.feature_importances_[FEATURE_COLS.index(f)]
    print(f"    {f:<35} {imp:.4f}")
print(f"  Ventanas train       : {len(X_train):,}")
print(f"  Ventanas test        : {len(X_test):,}")
print(f"  Peso clase ataque    : {weight_ataque}")
print("-" * 55)
print(f"  AUC-ROC              : {roc_auc_score(y_test, y_proba):.4f}")
print(f"  AUC-PR               : {average_precision_score(y_test, y_proba):.4f}")
print(f"  Tasa detección att   : {tp/(tp+fn)*100:.2f}%  (umbral 0.5)")
print(f"  Tasa falsa alarma    : {fp/(fp+tn)*100:.2f}%  (umbral 0.5)")
print("-" * 55)
print(f"  Falsos Negativos     : {fn:,}  ataques no detectados")
print(f"  Umbral óptimo (F1)   : {best['threshold']}")
print(f"  Recall con óptimo    : {best['recall_ataque']}%")
print(f"  Falsa alarma óptimo  : {best['falsa_alarma']}%")
print("=" * 55)